# Script 2 — Preparação & Engenharia de Features (V9 — Features Ricas)
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Esta versão (V8) expande a V7 com targets em 4 horizontes temporais:
1. **4 horizontes de target**: `_ITR_T1` (próx. trimestre), `_ITR_T2`, `_ITR_T3`, `_DFP` (próx. anual).
2. **Cortes temporais corrigidos**: treino ≤ 2023 | teste 2024–2025 | prospectivo ≥ 2026.
3. **Cascata de previsão habilitada**: ITR Q1/2026 → previsão Q2/Q3/DFP 2026 → DFP 2027–2029.
4. **Lags/rolls sobre macro**, **YoY sem exclusões**, **razões cruzadas**, **interações setor×macro** (da V7).


## Etapa 0 — Dependências, logging e configuração global

In [1]:
import logging, json, pickle, warnings
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 40)

logger = logging.getLogger('pipeline_preparacao')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)

_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
_sh  = logging.StreamHandler(); _sh.setLevel(logging.INFO); _sh.setFormatter(_fmt)
logger.addHandler(_sh)
_fh  = logging.FileHandler(PASTA_SAIDA / 'logs' / 'pipeline_preparacao.log', mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG); _fh.setFormatter(_fmt)
logger.addHandler(_fh)

# ── Parâmetros configuráveis ──────────────────────────────────────────────
LISTA_KPIS = [
    'margem_bruta','margem_ebit','margem_liquida','margem_ebitda',
    'roe','roa','liquidez_corrente','liquidez_imediata',
    'endividamento','alavancagem_de','div_liquida','cobertura_juros',
    'giro_ativo','fco_receita','fco_lucro','EBITDA',
    'FCF','margem_fcf','conversao_caixa',
]
# ── Colunas-fonte por variável financeira ────────────────────────────────
# Usado pela Etapa 8 para construir targets em 4 horizontes.
TARGET_COLS_SOURCE = {
    'DRE_3.01'   : 'Receita Líquida',
    'DRE_3.11'   : 'Lucro Líquido',
    'EBITDA'     : 'EBITDA calculado',
    'BPA_1'      : 'Ativo Total',
    'BPA_1.01'   : 'Ativo Circulante',
    'BPP_2.01'   : 'Passivo Circulante',
    'BPP_2.03'   : 'Patrimônio Líquido',
    'BPP_2'      : 'Passivo Total',
    'DFC_MI_6.01': 'Fluxo de Caixa Operacional',
}

# ── TARGET_COLS: 4 horizontes por variável ────────────────────────────────
# _ITR_T1 = próximo ITR (t+1 trimestre)
# _ITR_T2 = segundo ITR à frente (t+2 trimestres)
# _ITR_T3 = terceiro ITR à frente (t+3 trimestres)
# _DFP    = próxima DFP anual (equivalente ao target único da V7)
#
# Isso permite ao modelo prever qualquer horizonte e ao Script 5 fazer
# a cascata Q1→Q2→Q3→DFP 2026, e projeções anuais 2027–2029.
TARGET_COLS = {}
for _col, _desc in TARGET_COLS_SOURCE.items():
    TARGET_COLS[f'TARGET_{_col}_ITR_T1'] = _col   # próximo trimestre
    TARGET_COLS[f'TARGET_{_col}_ITR_T2'] = _col   # 2 trimestres à frente
    TARGET_COLS[f'TARGET_{_col}_ITR_T3'] = _col   # 3 trimestres à frente
    TARGET_COLS[f'TARGET_{_col}_DFP']    = _col   # próxima DFP anual

# LOG_TARGETS: skew elevado justifica log1p no treinamento
# V8: gerado automaticamente para cobrir os 4 horizontes por variável
_LOG_BASES = {
    'DRE_3.01', 'EBITDA', 'BPA_1', 'BPA_1.01',
    'BPP_2.01', 'BPP_2.03', 'BPP_2', 'DFC_MI_6.01',
}
LOG_TARGETS = {
    f'TARGET_{base}{sufixo}'
    for base in _LOG_BASES
    for sufixo in ('_ITR_T1', '_ITR_T2', '_ITR_T3', '_DFP')
}
LIMIAR_NULO    = 0.80
FATOR_WINSOR   = 3.0
MAX_COLS_YOY   = 22   # ampliado para cobrir EBITDA, FCF e div_liquida
CLIP_YOY       = 5.0
CORR_MIN       = 0.10
N_FEATURES_RFE = 30   # ampliado; Etapa 10 usa seleção multi-target
FRAC_TREINO    = 0.75
GAP_YOY_MIN    = 340
GAP_YOY_MAX    = 395

# Período de coleta macro: 2015-2025 completos
ANO_INICIO_MACRO = 2015
ANO_FIM_MACRO    = 2025

logger.info("Script 2 V6 (DFP+ITR) iniciado | pandas=%s", pd.__version__)

COLS_MACRO_FINAL = []  # preenchido pela Etapa 1B


2026-05-07 15:03:22 | INFO     | Script 2 V6 (DFP+ITR) iniciado | pandas=3.0.1


## Etapa 1 — Carregamento e validação

In [2]:
cam_parquet = PASTA_SAIDA / 'dataset_cvm_consolidado.parquet'
if not cam_parquet.exists():
    raise FileNotFoundError(
        f"Parquet não encontrado: {cam_parquet}\n"
        "Execute o Script 1 (01_cvm_processamento_V5.ipynb) antes de continuar."
    )

dataset = pd.read_parquet(cam_parquet)
logger.info("Dataset carregado: %d × %d", *dataset.shape)

dataset['DT_REFER']  = pd.to_datetime(dataset['DT_REFER'], errors='coerce', utc=False)
TZ_DATASET           = dataset['DT_REFER'].dt.tz
dataset['ANO']       = dataset['DT_REFER'].dt.year.astype('Int64')
dataset['TRIMESTRE'] = dataset['DT_REFER'].dt.quarter.astype('Int64')
dataset['MES']       = dataset['DT_REFER'].dt.month.astype('Int64')

COLS_OBR = ['CNPJ_CIA','NOME_CIA','SETOR','ANO','ORIGEM','DT_REFER']
faltando = [c for c in COLS_OBR if c not in dataset.columns]
if faltando:
    raise ValueError(f"Colunas obrigatórias ausentes: {faltando}")

n_emp = dataset['NOME_CIA'].nunique()
kpis_presentes = [k for k in LISTA_KPIS if k in dataset.columns]
kpis_ausentes  = [k for k in LISTA_KPIS if k not in dataset.columns]
if kpis_ausentes:
    logger.warning("KPIs ausentes: %s", kpis_ausentes)

for orig in ['DFP','ITR']:
    sub = dataset[dataset['ORIGEM']==orig]
    logger.info("%-3s: %d obs | %d empresas | anos %s",
                orig, len(sub), sub['NOME_CIA'].nunique(),
                sorted(sub['ANO'].dropna().astype(int).unique()))

print(f"\n{'='*60}")
print(f"  Dataset carregado | {dataset.shape[0]:,} × {dataset.shape[1]}")
print(f"  Empresas : {n_emp}/25 | DFP: {(dataset['ORIGEM']=='DFP').sum()} | ITR: {(dataset['ORIGEM']=='ITR').sum()}")
print(f"  KPIs     : {len(kpis_presentes)}/{len(LISTA_KPIS)} | TZ: {TZ_DATASET}")
print(f"{'='*60}")


2026-05-07 15:03:29 | INFO     | Dataset carregado: 987 × 772
2026-05-07 15:03:29 | INFO     | DFP: 230 obs | 25 empresas | anos [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
2026-05-07 15:03:29 | INFO     | ITR: 757 obs | 25 empresas | anos [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]



  Dataset carregado | 987 × 772
  Empresas : 25/25 | DFP: 230 | ITR: 757
  KPIs     : 19/19 | TZ: America/Sao_Paulo


## Etapa 1B — Enriquecimento com Dados Macroeconômicos e de Mercado

Integra variáveis de contexto externo ao dataset financeiro. Duas fontes:

**BCB/SGS:** SELIC (432), IPCA (433), Câmbio (1), PIB trimestral (4380).
Coleta **ano a ano** (2015–2025), 10 requisições por série — evita rejeição
por payload excessivo e delimita o intervalo ao período do estudo.

**Yahoo Finance via yfinance:** volatilidade 60d do EWZ como proxy de risco-Brasil.

**Alinhamento temporal:** `merge_asof` **individual por série** (direction='backward').
Cada série é mergeada separadamente no `df_key` — elimina o bug de cobertura 0%
causado por misturar séries de frequência diária (EWZ) com mensal (BCB) em um
único `pd.DataFrame`, onde o índice dominante (diário) deixava as séries mensais
com NaN em todas as linhas do merge.

**Fallback:** API indisponível → coluna preenchida com NaN. Pipeline não falha.

In [3]:
import requests, warnings
from datetime import datetime

# ── Tickers B3 por empresa ────────────────────────────────────────────────
TICKERS_B3 = {
    'Petrobras': 'PETR4.SA', 'Prio': 'PRIO3.SA', 'Ultrapar': 'UGPA3.SA',
    'Raizen': 'RAIZ4.SA', 'Vibra Energia': 'VBBR3.SA',
    'Engie Brasil': 'EGIE3.SA', 'Equatorial Energia': 'EQTL3.SA',
    'Taesa': 'TAEE11.SA', 'CPFL Energia': 'CPFE3.SA', 'ISA CTEEP': 'TRPL4.SA',
    'Lojas Renner': 'LREN3.SA', 'Magazine Luiza': 'MGLU3.SA',
    'Alpargatas': 'ALPA4.SA', 'Arezzo': 'ARZZ3.SA', 'Grupo Mateus': 'GMAT3.SA',
    'Vale': 'VALE3.SA', 'Suzano': 'SUZB3.SA', 'Klabin': 'KLBN11.SA',
    'Gerdau': 'GGBR4.SA', 'CSN Mineracao': 'CMIN3.SA',
    'WEG': 'WEGE3.SA', 'Totvs': 'TOTS3.SA', 'Positivo': 'POSI3.SA',
    'Intelbras': 'INTB3.SA', 'Brisanet': 'BRIT3.SA',
}

SERIES_BCB = {
    'macro_selic':   432,
    'macro_ipca':    433,
    'macro_cambio':  1,
    'macro_pib_tri': 4380,
}


def buscar_serie_bcb_anual(codigo, ano_inicio=ANO_INICIO_MACRO, ano_fim=ANO_FIM_MACRO):
    """
    Coleta a série BCB ano a ano (uma requisição por ano) cobrindo
    [01/01/ano_inicio .. 31/12/ano_fim].

    Estratégia:
    - Loop de ANO_INICIO_MACRO até ANO_FIM_MACRO inclusive
    - dataInicial = 01/01/ANO  |  dataFinal = 31/12/ANO
    - Evita rejeição por payload excessivo (406/413) e delimita
      o intervalo exatamente ao período do TCC
    - Accept: application/json previne 406 Not Acceptable
    - Fragmentos de cada ano são concatenados no final
    """
    fragmentos = []
    for ano in range(ano_inicio, ano_fim + 1):
        data_ini = f'01/01/{ano}'
        data_fim = f'31/12/{ano}'
        url = (
            f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados'
            f'?formato=json&dataInicial={data_ini}&dataFinal={data_fim}'
        )
        try:
            r = requests.get(url, timeout=20,
                             headers={'Accept': 'application/json'})
            r.raise_for_status()
            dados = r.json()
            if not dados:
                logger.debug('BCB %d | %d: sem registros', codigo, ano)
                continue
            df_ano = pd.DataFrame(dados)
            df_ano['data']  = pd.to_datetime(df_ano['data'], format='%d/%m/%Y')
            df_ano['valor'] = pd.to_numeric(df_ano['valor'], errors='coerce')
            fragmentos.append(df_ano)
            logger.debug('BCB %d | %d: %d obs', codigo, ano, len(df_ano))
        except Exception as e:
            logger.warning('BCB serie %d | ano %d indisponivel: %s', codigo, ano, e)

    if not fragmentos:
        return pd.Series(dtype=float)

    df_total = pd.concat(fragmentos, ignore_index=True)
    df_total = df_total.drop_duplicates('data').sort_values('data')
    return df_total.set_index('data')['valor']


def buscar_dados_mercado(ticker, data_inicio='2015-01-01', data_fim='2025-12-31'):
    """Retorna DataFrame com retorno_12m e volatilidade_60d para o ticker."""
    try:
        import yfinance as yf
        hist = yf.download(ticker, start=data_inicio, end=data_fim,
                           progress=False, auto_adjust=True)
        if hist.empty:
            return pd.DataFrame()
        if isinstance(hist.columns, pd.MultiIndex):
            hist.columns = hist.columns.droplevel(1)
        close   = hist['Close'].squeeze()
        ret_12m = close.pct_change(252).rename('retorno_12m')
        vol_60d = close.pct_change().rolling(60).std().rename('volatilidade_60d')
        return pd.concat([ret_12m, vol_60d], axis=1)
    except Exception as e:
        logger.warning('yfinance %s indisponivel: %s', ticker, e)
        return pd.DataFrame()


# ── Coleta das séries BCB (loop ano a ano) ────────────────────────────────
logger.info('Coletando series macro do BCB/SGS (%d–%d, ano a ano)...',
            ANO_INICIO_MACRO, ANO_FIM_MACRO)
series_bcb = {}
for nome_col, codigo in SERIES_BCB.items():
    s = buscar_serie_bcb_anual(codigo)
    if not s.empty:
        series_bcb[nome_col] = s
        logger.info('  %s: %d obs (%s a %s)',
                    nome_col, len(s),
                    s.index.min().date(), s.index.max().date())
    else:
        logger.warning('  %s: sem dados (fallback para NaN)', nome_col)

# ── CDS Brasil via yfinance ───────────────────────────────────────────────
logger.info('Coletando CDS Brasil (proxy EWZ)...')
df_ewz = buscar_dados_mercado('EWZ')
if not df_ewz.empty and 'volatilidade_60d' in df_ewz.columns:
    vol_ewz = df_ewz['volatilidade_60d'].copy()
    # Normalizar tz para naive antes de entrar no dict
    if vol_ewz.index.tz is not None:
        vol_ewz.index = vol_ewz.index.tz_convert(None)
    series_bcb['macro_vol_brasil'] = vol_ewz.rename('macro_vol_brasil')

# ── Alinhamento temporal — merge_asof INDIVIDUAL por série ───────────────
#
# CORREÇÃO CENTRAL:
# A abordagem anterior criava pd.DataFrame(series_bcb) misturando séries de
# frequências diferentes (BCB mensal + EWZ diário). O índice resultante era
# dominado pelas ~3000 datas diárias do EWZ; nas linhas do merge_asof com
# DT_MACRO diário, macro_ipca e macro_pib_tri apareciam como NaN (cobertura 0%).
#
# Solução: cada série é mergeada individualmente no df_key e o resultado é
# atribuído diretamente como nova coluna do dataset, sem interferência cruzada.
#
if series_bcb:
    # Vetor de datas de referência sem timezone (naive) para o merge
    dt_naive = dataset['DT_REFER'].dt.tz_localize(None)
    df_key   = pd.DataFrame({
        'DT_REFER_naive': dt_naive.values,
        'idx_orig':       dataset.index.tolist(),
    }).sort_values('DT_REFER_naive').reset_index(drop=True)

    cols_macro = []
    for col_name, serie in series_bcb.items():
        # Normalizar tz para naive
        s_naive = serie.copy()
        if s_naive.index.tz is not None:
            s_naive.index = s_naive.index.tz_convert(None)
        else:
            s_naive.index = pd.to_datetime(s_naive.index).tz_localize(None)

        # Construção do DataFrame à prova de versão de pandas:
        # Evita depender de reset_index() + rename, cujo comportamento com
        # index.name=None varia entre pandas 3.0.1 e 3.0.2 (coluna 0 vs 'index'),
        # causando KeyError ao tentar acessar merged[col_name] depois do merge_asof.
        # cast para datetime64[us] garante compatibilidade com df_key independente
        # da resolução retornada pelo BCB (pandas 3.0.1 retorna [s], 3.0.2 retorna [us])
        dt_macro = s_naive.index.astype('datetime64[us]')
        df_s = pd.DataFrame({'DT_MACRO': dt_macro, col_name: s_naive.values})
        df_s = df_s.sort_values('DT_MACRO').reset_index(drop=True)

        # Tolerância: 120 dias cobre PIB trimestral (~90 d de defasagem publicação)
        merged = pd.merge_asof(
            df_key[['DT_REFER_naive', 'idx_orig']],
            df_s,
            left_on='DT_REFER_naive',
            right_on='DT_MACRO',
            direction='backward',
            tolerance=pd.Timedelta('120 days'),
        ).set_index('idx_orig').sort_index()

        # Atribuir pelo índice (não por posição) — alinhamento seguro
        dataset[col_name] = merged[col_name]
        cob = dataset[col_name].notna().mean()
        logger.info('  Macro %-25s integrada | cobertura %.0f%%', col_name, cob * 100)
        cols_macro.append(col_name)

    logger.info('Macro integrada: %d colunas | cobertura media %.0f%%',
                len(cols_macro),
                dataset[cols_macro].notna().mean().mean() * 100 if cols_macro else 0)
else:
    cols_macro = []
    logger.warning('Nenhuma serie macro disponivel — pipeline continua sem dados macro')

COLS_MACRO_FINAL = cols_macro.copy()
print(f'Macro integrada: {len(cols_macro)} colunas')
if cols_macro:
    print(dataset[cols_macro].describe().round(4).to_string())


2026-05-07 15:03:32 | INFO     | Coletando series macro do BCB/SGS (2015–2025, ano a ano)...
2026-05-07 15:03:58 | INFO     |   macro_selic: 4018 obs (2015-01-01 a 2025-12-31)
2026-05-07 15:04:05 | INFO     |   macro_ipca: 132 obs (2015-01-01 a 2025-12-01)
2026-05-07 15:04:52 | INFO     |   macro_cambio: 2760 obs (2015-01-02 a 2025-12-31)
2026-05-07 15:05:02 | INFO     |   macro_pib_tri: 132 obs (2015-01-01 a 2025-12-01)
2026-05-07 15:05:02 | INFO     | Coletando CDS Brasil (proxy EWZ)...
2026-05-07 15:05:04 | INFO     |   Macro macro_selic               integrada | cobertura 100%
2026-05-07 15:05:04 | INFO     |   Macro macro_ipca                integrada | cobertura 100%
2026-05-07 15:05:04 | INFO     |   Macro macro_cambio              integrada | cobertura 100%
2026-05-07 15:05:04 | INFO     |   Macro macro_pib_tri             integrada | cobertura 100%
2026-05-07 15:05:04 | INFO     |   Macro macro_vol_brasil          integrada | cobertura 100%
2026-05-07 15:05:04 | INFO     | Mac

Macro integrada: 5 colunas
       macro_selic  macro_ipca  macro_cambio  macro_pib_tri  macro_vol_brasil
count     987.0000    987.0000      987.0000       987.0000          987.0000
mean        9.8323      0.5060        4.5825   738,114.3174            0.0202
std         4.2069      0.4355        0.9262   185,986.0211            0.0088
min         2.0000     -0.2900        3.1026   490,621.4000            0.0107
25%         6.5000      0.2100        3.8440   578,118.2000            0.0157
50%        10.7500      0.4800        4.9962   705,408.8000            0.0182
75%        13.7500      0.7300        5.4394   915,491.4000            0.0217
max        15.0000      1.6200        6.1923 1,082,332.3000            0.0644


In [4]:
# ── Dados de mercado por empresa (yfinance) ──────────────────────────────
logger.info('Coletando dados de mercado via yfinance...')

def buscar_mercado_anual(ticker, ano_inicio=ANO_INICIO_MACRO, ano_fim=ANO_FIM_MACRO):
    """
    Coleta dados de mercado ano a ano (uma requisição por ano).
    - Mesma estratégia do BCB: evita payload excessivo e delimita o período
    - Cast explícito para datetime64[us] garante compatibilidade com df_key
      no merge_asof (pandas 3.0.1 retorna [s], 3.0.2 retorna [us])
    - Retorna DataFrame com colunas [DT_MKT, retorno_12m, volatilidade_60d]
      ou DataFrame vazio se ticker indisponível/delistado
    """
    try:
        import yfinance as yf
        fragmentos = []
        for ano in range(ano_inicio, ano_fim + 1):
            ini = f'{ano}-01-01'
            fim = f'{ano}-12-31'
            hist = yf.download(ticker, start=ini, end=fim,
                               progress=False, auto_adjust=True)
            if hist.empty:
                continue
            if isinstance(hist.columns, pd.MultiIndex):
                hist.columns = hist.columns.droplevel(1)
            fragmentos.append(hist[['Close']].copy())

        if not fragmentos:
            return pd.DataFrame()

        close = pd.concat(fragmentos)['Close'].squeeze().sort_index()
        # Normalizar tz e resolução temporal de uma vez
        if close.index.tz is not None:
            close.index = close.index.tz_convert(None)
        # Cast para datetime64[us] — compatibilidade com _dt_naive do dataset
        close.index = close.index.astype('datetime64[us]')

        ret_12m = close.pct_change(252).rename('retorno_12m')
        vol_60d = close.pct_change().rolling(60).std().rename('volatilidade_60d')
        df = pd.concat([ret_12m, vol_60d], axis=1).reset_index()
        df = df.rename(columns={df.columns[0]: 'DT_MKT'})
        return df

    except Exception as e:
        logger.warning('yfinance %s indisponivel: %s', ticker, e)
        return pd.DataFrame()


if 'NOME_CIA' in dataset.columns:
    df_mercado_list = []
    for empresa, ticker in TICKERS_B3.items():
        df_mkt = buscar_mercado_anual(ticker)
        if df_mkt.empty:
            logger.warning('  Mercado %s (%s): sem dados', empresa, ticker)
            continue
        df_mkt['NOME_CIA'] = empresa
        df_mercado_list.append(df_mkt)
        logger.debug('  Mercado %s: %d obs', empresa, len(df_mkt))

    if df_mercado_list:
        df_mercado_all = pd.concat(df_mercado_list, ignore_index=True).sort_values('DT_MKT')

        partes_mkt = []
        for empresa, grp in dataset.sort_values('DT_REFER').groupby('NOME_CIA'):
            df_emp_mkt = df_mercado_all[df_mercado_all['NOME_CIA'] == empresa]
            if df_emp_mkt.empty:
                partes_mkt.append(grp)
                continue

            grp_c = grp.copy()
            # _dt_naive: tz_localize(None) remove tz preservando hora local (00:00)
            # cast para [us] garante compatibilidade com DT_MKT (também [us])
            grp_c['_dt_naive'] = (grp_c['DT_REFER']
                                  .dt.tz_localize(None)
                                  .astype('datetime64[us]'))

            merged = pd.merge_asof(
                grp_c.sort_values('_dt_naive'),
                df_emp_mkt[['DT_MKT', 'retorno_12m', 'volatilidade_60d']],
                left_on='_dt_naive', right_on='DT_MKT',
                direction='backward',
                tolerance=pd.Timedelta('31 days'),
            ).drop(columns=['DT_MKT', '_dt_naive'], errors='ignore')
            partes_mkt.append(merged)

        dataset = pd.concat(partes_mkt, ignore_index=True)
        dataset = dataset.sort_values(['CNPJ_CIA', 'DT_REFER']).reset_index(drop=True)

        cols_mkt = [c for c in dataset.columns if c in ['retorno_12m', 'volatilidade_60d']]
        for c in cols_mkt:
            if c not in cols_macro:
                cols_macro.append(c)
        logger.info('Dados de mercado integrados: %s | cobertura: %s',
                    cols_mkt,
                    {c: f"{dataset[c].notna().mean():.0%}" for c in cols_mkt})
        print(f'Dados de mercado integrados: {cols_mkt}')
    else:
        logger.warning('Dados de mercado indisponiveis — pipeline continua sem eles')
else:
    logger.warning('NOME_CIA nao encontrado — dados de mercado ignorados')

COLS_MACRO_FINAL = [c for c in dataset.columns
                    if c.startswith('macro_') or c in ['retorno_12m', 'volatilidade_60d']]
logger.info('Total colunas macro+mercado: %d', len(COLS_MACRO_FINAL))
print(f'Total colunas macro+mercado adicionadas: {len(COLS_MACRO_FINAL)}')


2026-05-07 15:05:09 | INFO     | Coletando dados de mercado via yfinance...
$UGPA3.SA: possibly delisted; no price data found  (1d 2025-01-01 -> 2025-12-31)

1 Failed download:
['UGPA3.SA']: possibly delisted; no price data found  (1d 2025-01-01 -> 2025-12-31)
$RAIZ4.SA: possibly delisted; no price data found  (1d 2015-01-01 -> 2015-12-31) (Yahoo error = "Data doesn't exist for startDate = 1420077600, endDate = 1451527200")

1 Failed download:
['RAIZ4.SA']: possibly delisted; no price data found  (1d 2015-01-01 -> 2015-12-31) (Yahoo error = "Data doesn't exist for startDate = 1420077600, endDate = 1451527200")
$RAIZ4.SA: possibly delisted; no price data found  (1d 2016-01-01 -> 2016-12-31) (Yahoo error = "Data doesn't exist for startDate = 1451613600, endDate = 1483149600")

1 Failed download:
['RAIZ4.SA']: possibly delisted; no price data found  (1d 2016-01-01 -> 2016-12-31) (Yahoo error = "Data doesn't exist for startDate = 1451613600, endDate = 1483149600")
$RAIZ4.SA: possibly delis

Dados de mercado integrados: ['retorno_12m', 'volatilidade_60d']
Total colunas macro+mercado adicionadas: 7


## Etapa 2 — Deduplicação intra-período

In [5]:
n_antes = len(dataset)
dataset['_n_kpis'] = dataset[kpis_presentes].notna().sum(axis=1)
dataset = (dataset
    .sort_values(['CNPJ_CIA','DT_REFER','ORIGEM','_n_kpis'], ascending=[True,True,True,False])
    .drop_duplicates(subset=['CNPJ_CIA','DT_REFER','ORIGEM'], keep='first')
    .drop(columns=['_n_kpis'])
    .sort_values(['CNPJ_CIA','DT_REFER'])
    .reset_index(drop=True)
)
rem = n_antes - len(dataset)
logger.info("Dedup: %d → %d (-%d)", n_antes, len(dataset), rem)
print(f"Dedup: {n_antes} → {len(dataset)} (removidas {rem} retificações)")


2026-05-07 15:09:22 | INFO     | Dedup: 987 → 987 (-0)


Dedup: 987 → 987 (removidas 0 retificações)


## Etapa 3 — Remoção de colunas com >80% de nulos

In [6]:
COLS_MACRO_FINAL = [c for c in dataset.columns
                    if c.startswith('macro_') or c in ['retorno_12m','volatilidade_60d']]
COLS_PROTEGIDAS = set(kpis_presentes + ['ANO','TRIMESTRE','MES'] + COLS_MACRO_FINAL)
cols_num        = dataset.select_dtypes(include='number').columns.tolist()
cols_cand       = [c for c in cols_num if c not in COLS_PROTEGIDAS]
taxa_nulo       = dataset[cols_cand].isnull().mean()
cols_excluir    = taxa_nulo[taxa_nulo > LIMIAR_NULO].index.tolist()

grupos_exc = {}
for c in cols_excluir:
    p = c.split('_')[0]; grupos_exc[p] = grupos_exc.get(p, 0) + 1
logger.info("Remoção >%.0f%% nulos: %d colunas | %s", LIMIAR_NULO*100, len(cols_excluir),
            dict(sorted(grupos_exc.items(), key=lambda x: -x[1])))

dataset = dataset.drop(columns=cols_excluir)
kpis_presentes = [k for k in kpis_presentes if k in dataset.columns]
print(f"Removidas: {len(cols_excluir)} colunas | Dataset: {dataset.shape} | KPIs: {len(kpis_presentes)}")


2026-05-07 15:10:10 | INFO     | Remoção >80% nulos: 328 colunas | {'BPP': 90, 'BPA': 77, 'DRE': 44, 'DVA': 37, 'DMPL': 35, 'DFC': 34, 'DRA': 11}


Removidas: 328 colunas | Dataset: (987, 451) | KPIs: 19



## Etapa 4 — Imputação e Winsorização aprendidas apenas no treino

Nesta etapa, os parâmetros de imputação e de truncamento de outliers são
estimados somente a partir do conjunto de treino temporal (até 2022).
Em seguida, esses parâmetros são aplicados ao dataset inteiro, incluindo
teste e prospectivo, sem usar informações dessas partições para aprender
estatísticas.

Isso evita vazamento temporal na preparação dos dados.

In [8]:
# Split temporal provisório apenas para aprender estatísticas de pré-processamento
ANO_CORTE_TREINO_PRE = 2022
ANO_CORTE_TESTE_PRE  = 2024

dataset['_split_pre'] = np.where(
    dataset['ANO'].astype(float) <= ANO_CORTE_TREINO_PRE, 'treino',
    np.where(dataset['ANO'].astype(float) <= ANO_CORTE_TESTE_PRE, 'teste', 'prospectivo')
)

treino_pre = dataset[dataset['_split_pre'] == 'treino'].copy()

# ---- Imputação por setor/origem aprendida apenas no treino -------------------
n_nulos_pre = dataset[kpis_presentes].isnull().sum().sum()

for kpi in kpis_presentes:
    if dataset[kpi].isnull().sum() == 0:
        continue

    med_setor_origem = treino_pre.groupby(['SETOR', 'ORIGEM'])[kpi].median()
    med_setor = treino_pre.groupby('SETOR')[kpi].median()
    med_global = treino_pre[kpi].median()

    idx = pd.MultiIndex.from_frame(dataset[['SETOR', 'ORIGEM']])

    # Preenche com mediana por setor+origem
    dataset[kpi] = dataset[kpi].fillna(pd.Series(idx.map(med_setor_origem), index=dataset.index))
    # Depois com mediana por setor
    dataset[kpi] = dataset[kpi].fillna(dataset['SETOR'].map(med_setor))
    # Por fim com mediana global do treino
    dataset[kpi] = dataset[kpi].fillna(med_global)

n_nulos_pos = dataset[kpis_presentes].isnull().sum().sum()
logger.info("Imputação (treino apenas): %d → %d nulos", n_nulos_pre, n_nulos_pos)
print(f"Nulos KPIs: {n_nulos_pre} → {n_nulos_pos}")

2026-05-07 15:11:53 | INFO     | Imputação (treino apenas): 0 → 0 nulos


Nulos KPIs: 0 → 0


## Etapa 5 — Winsorização por setor

Usa `groupby().transform()` — imune ao drop de `SETOR` do pandas 3.x.

In [9]:
# ---- Winsorização por setor aprendida apenas no treino ----------------------
def winsor_bounds_from_train(df_train, col, fator=3.0):
    bounds = {}
    for setor, grp in df_train.groupby('SETOR'):
        s = grp[col].dropna()
        if s.empty:
            continue
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            low, high = s.min(), s.max()
        else:
            low, high = q1 - fator * iqr, q3 + fator * iqr
        bounds[setor] = (low, high)

    s_all = df_train[col].dropna()
    if s_all.empty:
        global_bounds = (np.nan, np.nan)
    else:
        q1, q3 = s_all.quantile(0.25), s_all.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            global_bounds = (s_all.min(), s_all.max())
        else:
            global_bounds = (q1 - fator * iqr, q3 + fator * iqr)

    return bounds, global_bounds

n_clip_total = 0
winsor_info = {}

for kpi in kpis_presentes:
    bounds_setor, bounds_global = winsor_bounds_from_train(treino_pre, kpi, FATOR_WINSOR)
    winsor_info[kpi] = {'setor': bounds_setor, 'global': bounds_global}

    low_g, high_g = bounds_global
    if pd.notna(low_g) and pd.notna(high_g):
        dataset[kpi] = dataset[kpi].clip(low_g, high_g)

    for setor, (low, high) in bounds_setor.items():
        mask = dataset['SETOR'] == setor
        antes = dataset.loc[mask, kpi].copy()
        dataset.loc[mask, kpi] = dataset.loc[mask, kpi].clip(low, high)
        n_clip_total += (dataset.loc[mask, kpi] != antes).sum()

dataset = dataset.drop(columns=['_split_pre'])
assert 'SETOR' in dataset.columns, "SETOR perdido após winsorização"

logger.info("Winsorização (treino apenas): fator=%.1f | %d valores truncados", FATOR_WINSOR, n_clip_total)
print(f"✅ Winsorização | fator={FATOR_WINSOR} | {n_clip_total} valores truncados | SETOR: ✅")

2026-05-07 15:11:59 | INFO     | Winsorização (treino apenas): fator=3.0 | 315 valores truncados


✅ Winsorização | fator=3.0 | 315 valores truncados | SETOR: ✅


## Etapa 6 — YoY mesmo trimestre ano anterior (shift=4)

In [10]:
dataset = dataset.sort_values(['CNPJ_CIA','DT_REFER']).reset_index(drop=True)

# V7: removida exclusão arbitrária de EBITDA, FCF e div_liquida —
# esses três são economicamente relevantes e têm targets associados.
KPIS_YOY = kpis_presentes[:MAX_COLS_YOY]

dataset['_dt_prev'] = dataset.groupby('CNPJ_CIA')['DT_REFER'].shift(4)
dataset['_gap_dias'] = (dataset['DT_REFER'] - dataset['_dt_prev']).dt.days
gap_invalido = ~dataset['_gap_dias'].between(GAP_YOY_MIN, GAP_YOY_MAX)
logger.info("YoY gaps inválidos: %d / %d (%.1f%%)",
            gap_invalido.sum(), len(dataset), gap_invalido.sum()/len(dataset)*100)

cols_yoy = []
for kpi in KPIS_YOY:
    col_yoy = f'{kpi}_yoy'
    prev    = dataset.groupby('CNPJ_CIA')[kpi].shift(4)
    yoy_raw = ((dataset[kpi] - prev) / prev.abs().replace(0, np.nan))
    yoy_raw = yoy_raw.replace([np.inf,-np.inf], np.nan).clip(-CLIP_YOY, CLIP_YOY)
    yoy_raw[gap_invalido] = np.nan
    dataset[col_yoy] = yoy_raw
    cols_yoy.append(col_yoy)

if 'margem_ebitda_yoy' in dataset.columns:
    accel = dataset.groupby('CNPJ_CIA')['margem_ebitda_yoy'].diff()
    accel[gap_invalido] = np.nan
    dataset['aceleracao_ebitda'] = accel
    cols_yoy.append('aceleracao_ebitda')

dataset['pos_ciclo'] = dataset['TRIMESTRE'].astype(float)
dataset.loc[dataset['ORIGEM']=='DFP', 'pos_ciclo'] = 4.0
cols_yoy.append('pos_ciclo')

dataset = dataset.drop(columns=['_dt_prev','_gap_dias'])

logger.info("YoY: %d features | nulos médios: %.0f%%",
            len(cols_yoy), dataset[cols_yoy].isnull().mean().mean()*100)
print(f"Features YoY: {len(cols_yoy)} | nulos médios: {dataset[cols_yoy].isnull().mean().mean():.0%}")


2026-05-07 15:13:26 | INFO     | YoY gaps inválidos: 111 / 987 (11.2%)
2026-05-07 15:13:26 | INFO     | YoY: 18 features | nulos médios: 11%


Features YoY: 18 | nulos médios: 11%


## Etapa 7 — One-hot do setor e flag de origem

In [11]:
dataset = pd.get_dummies(dataset, columns=['SETOR'], prefix='setor', dtype=float)
cols_setor = sorted([c for c in dataset.columns if c.startswith('setor_')])
dataset['flag_dfp'] = (dataset['ORIGEM'] == 'DFP').astype(float)

# ── Sazonalidade trimestral explícita ────────────────────────────────────
# Empresas têm padrões sazonais fortes (ex: varejo no Q4, agro no Q1).
# Dummies de trimestre capturam isso sem depender de lags.
if 'DT_REFER' in dataset.columns:
    dataset['trimestre'] = pd.to_datetime(dataset['DT_REFER']).dt.quarter
    dataset = pd.get_dummies(dataset, columns=['trimestre'], prefix='tri', dtype=float)
    cols_tri = sorted([c for c in dataset.columns if c.startswith('tri_')])
else:
    cols_tri = []
logger.info("Sazonalidade trimestral: %d dummies criadas", len(cols_tri))

logger.info("One-hot SETOR: %d colunas | flag_dfp criada", len(cols_setor))
print(f"Setores: {cols_setor}")
print(f"Dataset: {dataset.shape}")


2026-05-07 15:13:30 | INFO     | One-hot SETOR: 5 colunas | flag_dfp criada


Setores: ['setor_Commodities', 'setor_Energia', 'setor_Petróleo', 'setor_Tecnologia', 'setor_Varejo']
Dataset: (987, 474)


## Etapa 8 — Construção dos targets prospectivos

Nesta etapa, o dataset consolidado recebe as colunas-alvo do problema de predição.
Para cada linha, busca-se o **próximo DFP estritamente posterior** da mesma companhia,
de forma que os targets representem o valor futuro a ser previsto a partir da
informação disponível na data corrente.

O procedimento preserva a ordenação temporal por `CNPJ_CIA` e `DT_REFER`, e não
elimina nenhuma observação nesta fase. As linhas sem target conhecido são
mantidas para compor o conjunto prospectivo, que será consumido pelo Script 5.

Guard para `TZ_DATASET=None`: evita `TypeError: Invalid datetime unit in metadata string "[us, None]"`.

In [12]:
# =============================================================================
# Etapa 8 — Targets em 4 horizontes: ITR_T1, ITR_T2, ITR_T3, DFP  (V8)
# =============================================================================
#
# Para cada linha do dataset (DFP ou ITR), criamos 4 targets por variável:
#   _ITR_T1 : valor do próximo ITR publicado após DT_REFER
#   _ITR_T2 : valor do 2º ITR publicado após DT_REFER
#   _ITR_T3 : valor do 3º ITR publicado após DT_REFER
#   _DFP    : valor da próxima DFP publicada após DT_REFER
#
# Isso habilita o modelo a prever qualquer horizonte e o Script 5 a fazer
# a cascata Q1→Q2→Q3→DFP 2026, além de projeções anuais 2027–2029.

# Fontes separadas por tipo de documento
itr_fonte = (
    dataset[dataset['ORIGEM'] == 'ITR']
    [['CNPJ_CIA', 'DT_REFER'] + list(TARGET_COLS_SOURCE.keys())]
    .sort_values(['CNPJ_CIA', 'DT_REFER'])
    .reset_index(drop=True)
)
dfp_fonte = (
    dataset[dataset['ORIGEM'] == 'DFP']
    [['CNPJ_CIA', 'DT_REFER'] + list(TARGET_COLS_SOURCE.keys())]
    .sort_values(['CNPJ_CIA', 'DT_REFER'])
    .reset_index(drop=True)
)

# Inicializa todas as colunas-target como NaN
all_target_cols = list(TARGET_COLS.keys())
for col in all_target_cols:
    dataset[col] = np.nan

# DT_TARGET_DFP: data da próxima DFP (para rastreabilidade e anti-contaminação)
# Inicializa como object para evitar conflito tz-naive vs tz-aware durante o loop
# Será convertido para datetime64[ns] após a concatenação
dataset['DT_TARGET_DFP'] = None

partes = []
for cnpj, grupo in dataset.sort_values(['CNPJ_CIA', 'DT_REFER']).groupby('CNPJ_CIA'):
    itr_emp = itr_fonte[itr_fonte['CNPJ_CIA'] == cnpj].sort_values('DT_REFER').reset_index(drop=True)
    dfp_emp = dfp_fonte[dfp_fonte['CNPJ_CIA'] == cnpj].sort_values('DT_REFER').reset_index(drop=True)

    g = grupo.sort_values('DT_REFER').copy()

    for idx in g.index:
        dt = g.loc[idx, 'DT_REFER']

        # ── Horizontes ITR (T1, T2, T3) ─────────────────────────────────
        itrs_futuros = itr_emp[itr_emp['DT_REFER'] > dt].reset_index(drop=True)
        for t_offset, sufixo in enumerate(['_ITR_T1', '_ITR_T2', '_ITR_T3']):
            if t_offset < len(itrs_futuros):
                linha_itr = itrs_futuros.iloc[t_offset]
                for col_src in TARGET_COLS_SOURCE:
                    tgt_name = f'TARGET_{col_src}{sufixo}'
                    if tgt_name in g.columns:
                        g.loc[idx, tgt_name] = linha_itr[col_src]

        # ── Horizonte DFP ────────────────────────────────────────────────
        dfps_futuras = dfp_emp[dfp_emp['DT_REFER'] > dt].reset_index(drop=True)
        if len(dfps_futuras):
            proxima_dfp = dfps_futuras.iloc[0]
            for col_src in TARGET_COLS_SOURCE:
                tgt_name = f'TARGET_{col_src}_DFP'
                if tgt_name in g.columns:
                    g.loc[idx, tgt_name] = proxima_dfp[col_src]
            # Guarda data da DFP alvo para rastreabilidade
            # Normaliza timezone: remove tz-info para compatibilidade com datetime64[ns]
            dt_dfp = proxima_dfp['DT_REFER']
            if hasattr(dt_dfp, 'tzinfo') and dt_dfp.tzinfo is not None:
                dt_dfp = dt_dfp.tz_convert('UTC').tz_localize(None)
            g.loc[idx, 'DT_TARGET_DFP'] = dt_dfp

    partes.append(g)

dataset = pd.concat(partes, ignore_index=True)
dataset = dataset.sort_values(['CNPJ_CIA', 'DT_REFER']).reset_index(drop=True)

# Converte DT_TARGET_DFP para datetime64[ns] sem timezone
dataset['DT_TARGET_DFP'] = pd.to_datetime(dataset['DT_TARGET_DFP'], utc=True, errors='coerce').dt.tz_localize(None)

targets_criados = [t for t in TARGET_COLS if t in dataset.columns]

# Resumo por horizonte
logger.info("Targets criados: %d colunas (%d variáveis × 4 horizontes)", len(targets_criados), len(TARGET_COLS_SOURCE))
print(f"\nTargets criados: {len(targets_criados)} ({len(TARGET_COLS_SOURCE)} variáveis × 4 horizontes)")
print(f"{'Horizonte':<12} {'DFP com target':>15} {'ITR com target':>15}")
print("-" * 45)
for sufixo in ['_ITR_T1', '_ITR_T2', '_ITR_T3', '_DFP']:
    cols_h = [t for t in targets_criados if t.endswith(sufixo)]
    n_dfp = dataset[dataset['ORIGEM']=='DFP'][cols_h[0]].notna().sum() if cols_h else 0
    n_itr = dataset[dataset['ORIGEM']=='ITR'][cols_h[0]].notna().sum() if cols_h else 0
    print(f"  {sufixo:<12} {n_dfp:>15,} {n_itr:>15,}")


2026-05-07 15:13:38 | INFO     | Targets criados: 9


Targets criados:
  DFP: 205/230 obs com target
  ITR: 676/757 obs com target


## Etapa 8B — Lags e memória temporal por empresa

Aqui são criadas variáveis temporais por companhia para capturar persistência,
tendência e aceleração dos indicadores. As features são construídas sempre com
base na ordem `CNPJ_CIA` + `DT_REFER`, usando deslocamento temporal (`shift`)
antes de qualquer média móvel ou cálculo de crescimento.

Essa etapa é feita sobre o dataset completo, para que o conjunto prospectivo
também receba as mesmas transformações estruturais aplicadas ao restante da base.

In [13]:
# =============================================================================
# Etapa 8B — Lags e memória temporal por empresa
# =============================================================================

LAG_PERIODS = (1, 2, 4)
ROLL_WINDOWS = (2, 4)

def criar_features_temporais_por_empresa(
    df,
    group_col='CNPJ_CIA',
    time_col='DT_REFER',
    base_cols=None,
    lag_periods=LAG_PERIODS,
    roll_windows=ROLL_WINDOWS,
):
    df = df.sort_values([group_col, time_col]).copy()
    cols_criadas = []

    if base_cols is None:
        base_cols = []

    # Mantém apenas colunas numéricas que realmente existem
    base_cols = [
        c for c in base_cols
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
    ]

    for col in base_cols:
        g = df.groupby(group_col, sort=False)[col]
        prev = g.shift(1)

        # Lags explícitos
        for lag in lag_periods:
            nome = f'{col}_lag{lag}'
            df[nome] = g.shift(lag)
            cols_criadas.append(nome)

        # Diferença absoluta em relação ao período anterior
        nome_diff = f'{col}_diff1'
        df[nome_diff] = df[col] - prev
        cols_criadas.append(nome_diff)

        # Taxa de crescimento em relação ao período anterior
        nome_growth = f'{col}_growth1'
        denom = prev.abs().replace(0, np.nan)
        df[nome_growth] = (df[col] - prev) / denom
        cols_criadas.append(nome_growth)

        # Médias móveis sempre com shift(1), para não usar o valor atual
        for w in roll_windows:
            nome_roll = f'{col}_roll{w}_mean'
            df[nome_roll] = g.transform(
                lambda s: s.shift(1).rolling(w, min_periods=2).mean()
            )
            cols_criadas.append(nome_roll)

    return df, cols_criadas


# Base para memória temporal:
# V7: inclui KPIs, targets e séries macro — capturam tendência e aceleração macro.
# Lags sobre macro (lag1, lag2, diff1, roll4) são features comprovadas em finanças.
BASE_LAG_COLS = list(dict.fromkeys(
    [c for c in kpis_presentes if c in dataset.columns]
    + targets_criados
    + [c for c in COLS_MACRO_FINAL if c in dataset.columns]
))

dataset, cols_lag = criar_features_temporais_por_empresa(
    dataset,
    group_col='CNPJ_CIA',
    time_col='DT_REFER',
    base_cols=BASE_LAG_COLS,
    lag_periods=LAG_PERIODS,
    roll_windows=ROLL_WINDOWS,
)

dataset = dataset.sort_values(['CNPJ_CIA', 'DT_REFER']).reset_index(drop=True)

logger.info("Features temporais criadas: %d", len(cols_lag))
print(f"Features temporais criadas: {len(cols_lag)}")
print(f"Exemplos: {cols_lag[:10]}")

2026-05-07 15:13:54 | INFO     | Features temporais criadas: 196


Features temporais criadas: 196
Exemplos: ['margem_bruta_lag1', 'margem_bruta_lag2', 'margem_bruta_lag4', 'margem_bruta_diff1', 'margem_bruta_growth1', 'margem_bruta_roll2_mean', 'margem_bruta_roll4_mean', 'margem_ebit_lag1', 'margem_ebit_lag2', 'margem_ebit_lag4']


## Etapa 9 — Consolidação das features candidatas

Nesta etapa, são reunidas todas as variáveis que podem entrar na modelagem:
KPIs, variações YoY, lags temporais, variáveis de setor, flags de origem e
séries macroeconômicas/mercado.

A seleção final das variáveis não é feita aqui. Ela será executada apenas após
o split temporal, usando somente o conjunto de treino, para evitar vazamento
de informação do teste na escolha das features.

In [14]:
# =============================================================================
# Etapa 9 — Consolidação das features candidatas
# =============================================================================

# =============================================================================
# Etapa 9B — Razões cruzadas fundamentalistas e interações setor × macro  (V7)
# =============================================================================

cols_razoes = []

# ── 1. Razões cruzadas fundamentalistas ─────────────────────────────────────
# Qualidade do lucro: FCO como fração do lucro líquido
if 'fco_lucro' in dataset.columns and 'margem_liquida' in dataset.columns:
    dataset['ratio_qualidade_lucro'] = (
        dataset['fco_lucro'] / dataset['margem_liquida'].abs().replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).clip(-10, 10)
    cols_razoes.append('ratio_qualidade_lucro')

# Cobertura de dívida: EBITDA / dívida líquida (alavancagem operacional)
if 'EBITDA' in dataset.columns and 'div_liquida' in dataset.columns:
    dataset['ratio_cobertura_divida'] = (
        dataset['EBITDA'] / dataset['div_liquida'].abs().replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).clip(-20, 20)
    cols_razoes.append('ratio_cobertura_divida')

# Alavancagem implícita: ROE / ROA
if 'roe' in dataset.columns and 'roa' in dataset.columns:
    dataset['ratio_alavancagem_impl'] = (
        dataset['roe'] / dataset['roa'].abs().replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).clip(-20, 20)
    cols_razoes.append('ratio_alavancagem_impl')

# Conversão de margem: FCO / Receita vs Margem EBITDA
if 'fco_receita' in dataset.columns and 'margem_ebitda' in dataset.columns:
    dataset['ratio_conversao_ebitda'] = (
        dataset['fco_receita'] / dataset['margem_ebitda'].abs().replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).clip(-10, 10)
    cols_razoes.append('ratio_conversao_ebitda')

# Eficiência relativa: giro do ativo normalizado por endividamento
if 'giro_ativo' in dataset.columns and 'endividamento' in dataset.columns:
    dataset['ratio_efic_endiv'] = (
        dataset['giro_ativo'] / dataset['endividamento'].abs().replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).clip(-20, 20)
    cols_razoes.append('ratio_efic_endiv')

logger.info("Razões cruzadas criadas: %d", len(cols_razoes))

# ── 2. Interações setor × macro ──────────────────────────────────────────────
cols_interacao = []
cols_setor_atual = [c for c in dataset.columns if c.startswith('setor_')]
macro_interacao = [c for c in ['macro_cambio', 'macro_ipca', 'macro_selic']
                   if c in dataset.columns]

for col_macro in macro_interacao:
    for col_setor in cols_setor_atual:
        nome_int = f'int_{col_setor}_{col_macro}'
        dataset[nome_int] = dataset[col_setor] * dataset[col_macro]
        cols_interacao.append(nome_int)

logger.info("Interações setor×macro criadas: %d (%d setores × %d séries macro)",
            len(cols_interacao), len(cols_setor_atual), len(macro_interacao))
print(f"Razões cruzadas: {len(cols_razoes)} | Interações setor×macro: {len(cols_interacao)}")

# ── 3. Features de proporção (targets como crescimento relativo) ─────────
# Em vez de valor absoluto, modelamos a variação relativa.
# Ex: crescimento_DRE_3.01 = (DRE_3.01_t - DRE_3.01_{t-1}) / |DRE_3.01_{t-1}|
# Remove a diferença de escala entre empresas sem demeaning — cada empresa
# é comparada consigo mesma. Fundamentado em Fama & French (1992).
cols_proporcao = []
for col_src in TARGET_COLS_SOURCE:
    if col_src not in dataset.columns:
        continue
    col_prev = dataset.groupby('CNPJ_CIA')[col_src].shift(1)
    col_prev_abs = col_prev.abs().replace(0, np.nan)
    nome_prop = f'prop_{col_src}'
    dataset[nome_prop] = ((dataset[col_src] - col_prev) / col_prev_abs).clip(-5, 5)
    cols_proporcao.append(nome_prop)
logger.info("Features de proporção criadas: %d", len(cols_proporcao))

# ── 4. Posição relativa no setor ─────────────────────────────────────────
# Ex: receita_empresa / mediana_receita_setor — mede posicionamento competitivo.
# Captura se a empresa está acima ou abaixo da mediana do setor, o que
# tem poder preditivo sobre crescimento futuro (mean reversion setorial).
cols_pos_setor = []
if 'SETOR' in dataset.columns or any(c.startswith('setor_') for c in dataset.columns):
    # Reconstrói SETOR a partir das dummies se necessário
    _cols_s = [c for c in dataset.columns if c.startswith('setor_')]
    if _cols_s and 'SETOR' not in dataset.columns:
        dataset['_SETOR_TMP'] = dataset[_cols_s].idxmax(axis=1).str.replace('setor_', '')
        setor_col = '_SETOR_TMP'
    else:
        setor_col = 'SETOR'

    for col_src in list(TARGET_COLS_SOURCE.keys())[:6]:  # top 6 variáveis-fonte
        if col_src not in dataset.columns:
            continue
        mediana_setor = dataset.groupby([setor_col, 'DT_REFER'])[col_src].transform('median')
        mediana_setor_abs = mediana_setor.abs().replace(0, np.nan)
        nome_pos = f'pos_setor_{col_src}'
        dataset[nome_pos] = (dataset[col_src] / mediana_setor_abs).clip(-10, 10)
        cols_pos_setor.append(nome_pos)

    if '_SETOR_TMP' in dataset.columns:
        dataset.drop(columns=['_SETOR_TMP'], inplace=True)

logger.info("Posição relativa no setor: %d features", len(cols_pos_setor))
print(f"Proporções: {len(cols_proporcao)} | Posição setor: {len(cols_pos_setor)}")

# =============================================================================
# Etapa 9 — Consolidação das features candidatas
# =============================================================================

FEATURES_CANDIDATAS = list(dict.fromkeys(
    kpis_presentes
    + cols_yoy
    + cols_lag
    + cols_setor
    + ['flag_dfp']
    + COLS_MACRO_FINAL
    + cols_razoes          # V7: razões cruzadas fundamentalistas
    + cols_interacao       # V7: interações setor × macro
    + cols_proporcao       # V9: targets como crescimento relativo
    + cols_pos_setor       # V9: posição relativa no setor
    + cols_tri             # V9: sazonalidade trimestral
))

# Filtra apenas colunas que existem de fato no dataset
FEATURES_CANDIDATAS = [c for c in FEATURES_CANDIDATAS if c in dataset.columns]

logger.info("Features candidatas totais: %d", len(FEATURES_CANDIDATAS))
print(f"\nFeatures candidatas: {len(FEATURES_CANDIDATAS)}")


2026-05-07 15:14:01 | INFO     | Features candidatas: 246


Features candidatas consolidadas: 246
Exemplos: ['margem_bruta', 'margem_ebit', 'margem_liquida', 'margem_ebitda', 'roe', 'roa', 'liquidez_corrente', 'liquidez_imediata', 'endividamento', 'alavancagem_de', 'div_liquida', 'cobertura_juros', 'giro_ativo', 'fco_receita', 'fco_lucro']



## Etapa 10 — Split temporal e seleção final de features

O dataset é dividido em treino, teste e prospectivo com base no ano de
referência. A seleção final de features é feita somente com o conjunto de
treino, evitando que o teste influencie a escolha das variáveis.

O conjunto prospectivo é preservado integralmente para o Script 5.

In [15]:
ANO_CORTE_TREINO = 2023   # treino: ≤ 2023 (V8: +1 ano vs V7)
ANO_CORTE_TESTE  = 2025   # teste:  2024–2025 | prospectivo: ≥ 2026 (ITR Q1 + previsão)

dataset['split'] = np.where(
    dataset['ANO'].astype(float) <= ANO_CORTE_TREINO,
    'treino',
    np.where(
        dataset['ANO'].astype(float) <= ANO_CORTE_TESTE,
        'teste',
        'prospectivo'
    )
)

assert 'CNPJ_CIA' in dataset.columns, "CNPJ_CIA perdido — invariante violado"

treino = dataset[dataset['split'] == 'treino'].copy()
teste = dataset[dataset['split'] == 'teste'].copy()
prospectivo = dataset[dataset['split'] == 'prospectivo'].copy()

# Mantém apenas linhas com ao menos um target conhecido em treino/teste
# V8: targets_criados cobre os 4 horizontes — any() é suficiente
targets_criados = [t for t in TARGET_COLS if t in dataset.columns]
treino = treino[treino[targets_criados].notna().any(axis=1)].copy()
teste  = teste[teste[targets_criados].notna().any(axis=1)].copy()

# Resumo de cobertura por horizonte no treino
print("\nCobertura de targets no treino:")
for sufixo in ['_ITR_T1', '_ITR_T2', '_ITR_T3', '_DFP']:
    cols_h = [t for t in targets_criados if t.endswith(sufixo)]
    if cols_h:
        n = treino[cols_h].notna().any(axis=1).sum()
        print(f"  {sufixo:<12}: {n:>4} obs com target")

# Verificação de contaminação temporal por empresa
contaminados = []
for cnpj, grp in dataset.groupby('CNPJ_CIA'):
    tr_max = grp[grp['split'] == 'treino']['DT_REFER'].max()
    te_min = grp[grp['split'] == 'teste']['DT_REFER'].min()
    if pd.notna(tr_max) and pd.notna(te_min) and tr_max > te_min:
        contaminados.append(cnpj)

if contaminados:
    logger.error("Contaminação temporal detectada: %s", contaminados)
else:
    logger.info("Anti-contaminação temporal: PASSOU ✅")

n_tot = len(dataset)
logger.info(
    "Split: treino=%d (%.0f%%) | teste=%d (%.0f%%) | prospectivo=%d (%.0f%%)",
    len(treino), len(treino) / n_tot * 100,
    len(teste), len(teste) / n_tot * 100,
    len(prospectivo), len(prospectivo) / n_tot * 100
)

print(f"\n{'='*60}")
print(f"  Split temporal — 3 conjuntos")
print(f"{'='*60}")
print(f"  Treino      (≤{ANO_CORTE_TREINO}): {len(treino):>4} obs | "
      f"DFP={(treino['ORIGEM']=='DFP').sum():>3} | "
      f"ITR={(treino['ORIGEM']=='ITR').sum():>3}")
print(f"  Teste    ({ANO_CORTE_TREINO+1}–{ANO_CORTE_TESTE}): {len(teste):>4} obs | "
      f"DFP={(teste['ORIGEM']=='DFP').sum():>3} | "
      f"ITR={(teste['ORIGEM']=='ITR').sum():>3}")
print(f"  Prospectivo (≥{ANO_CORTE_TESTE+1}): {len(prospectivo):>4} obs | "
      f"DFP={(prospectivo['ORIGEM']=='DFP').sum():>3} | "
      f"ITR={(prospectivo['ORIGEM']=='ITR').sum():>3}")
print(f"{'='*60}")
print(f"  Anti-contaminação: {'✅' if not contaminados else '❌'}")
print(f"  CNPJ_CIA preservado: ✅")
print()

if not prospectivo.empty:
    anos_prosp = sorted(prospectivo['ANO'].dropna().astype(int).unique())
    emps_prosp = prospectivo['NOME_CIA'].nunique()
    print(f"  Prospectivo cobre: {emps_prosp} empresas | anos {anos_prosp}")
    print(f"  → Será usado pelo Script 5 (predição genuína sem target conhecido)")
else:
    print("  ℹ️  Prospectivo vazio — verifique se há ITR/DFP recentes na base")

# ── Seleção multi-target — feita SOMENTE no treino  (V7) ────────────────────
#
# Problema V6: RFE com único target (TARGET_DRE_3.01) descartava features
# relevantes para EBITDA, FCO, Balanço, etc., antes de chegar ao Script 3.
#
# Solução V7: para cada target, seleciona as top-N features por correlação
# de Pearson e une os conjuntos. O Script 3 faz a poda final por família
# de modelo (threshold linear vs tree), então aqui o objetivo é cobrir
# amplamente — sem deixar nenhum target sem representação.

FEATURES_SELECIONADAS_UNION = set()
corr_por_target = {}

for tgt in targets_criados:
    cols_disp = [f for f in FEATURES_CANDIDATAS if f in treino.columns and f != tgt]
    df_sel = treino[cols_disp + [tgt]].dropna(subset=[tgt])
    if df_sel.empty or len(df_sel) < 5:
        logger.warning("Seleção multi-target: sem dados para %s — ignorado", tgt)
        continue

    corr_abs = df_sel[cols_disp].corrwith(df_sel[tgt]).abs().fillna(0).sort_values(ascending=False)
    corr_por_target[tgt] = corr_abs

    # Seleciona features com correlação mínima OU top-N_FEATURES_RFE (o que for maior)
    acima_min = corr_abs[corr_abs >= CORR_MIN].index.tolist()
    top_n     = corr_abs.head(N_FEATURES_RFE).index.tolist()
    selecionadas_tgt = list(dict.fromkeys(acima_min + top_n))  # union, ordem preservada
    FEATURES_SELECIONADAS_UNION.update(selecionadas_tgt)
    logger.info("  %-30s: %d features (|r|≥%.2f ou top-%d)",
                tgt, len(selecionadas_tgt), CORR_MIN, N_FEATURES_RFE)

# Ordena pela correlação média entre todos os targets onde a feature apareceu
def _corr_media(feat):
    vals = [corr_por_target[t][feat] for t in corr_por_target if feat in corr_por_target[t].index]
    return float(np.mean(vals)) if vals else 0.0

FEATURES_SELECIONADAS = sorted(FEATURES_SELECIONADAS_UNION, key=_corr_media, reverse=True)
FEATURES_SELECIONADAS = [f for f in FEATURES_SELECIONADAS if f in treino.columns]

logger.info("Seleção multi-target (treino apenas): %d candidatas → %d selecionadas",
            len(FEATURES_CANDIDATAS), len(FEATURES_SELECIONADAS))
print(f"\nFeatures selecionadas (multi-target): {len(FEATURES_SELECIONADAS)}")
print(f"  Candidatas originais : {len(FEATURES_CANDIDATAS)}")
print(f"  Após seleção union   : {len(FEATURES_SELECIONADAS)}")
print(f"  Targets cobertos     : {len(corr_por_target)}/{len(targets_criados)}")

# Salva prospectivo com todas as colunas já preparadas
if not prospectivo.empty:
    cam_prosp = PASTA_SAIDA / 'dataset_prospectivo.parquet'
    prospectivo.to_parquet(cam_prosp, index=False)
    logger.info("Prospectivo salvo: %d obs | %s", len(prospectivo), cam_prosp.name)

2026-05-07 15:14:06 | INFO     | Anti-contaminação temporal: PASSOU ✅
2026-05-07 15:14:06 | INFO     | Split: treino=713 (72%) | teste=168 (17%) | prospectivo=76 (8%)
2026-05-07 15:14:06 | INFO     | Pearson (treino apenas): 246 → 118 features
2026-05-07 15:14:06 | INFO     | RFE (treino apenas): 118 → 15 features



  Split temporal — 3 conjuntos
  Treino      (≤2022):  713 obs | DFP=181 | ITR=532
  Teste    (2023–2024):  168 obs | DFP= 24 | ITR=144
  Prospectivo (≥2025):   76 obs | DFP=  1 | ITR= 75
  Anti-contaminação: ✅
  CNPJ_CIA preservado: ✅

  Prospectivo cobre: 24 empresas | anos [np.int64(2025), np.int64(2026)]
  → Será usado pelo Script 5 (predição genuína sem target conhecido)

Features selecionadas (15):
  TARGET_DRE_3.01_lag1                |r| = 0.987
  TARGET_DRE_3.01_roll2_mean          |r| = 0.985
  TARGET_DRE_3.01_roll4_mean          |r| = 0.980
  TARGET_BPP_2.01_lag1                |r| = 0.937
  TARGET_BPP_2.01_lag2                |r| = 0.936
  TARGET_DFC_MI_6.01_lag1             |r| = 0.928
  TARGET_BPA_1.01_lag2                |r| = 0.925
  TARGET_BPP_2.03_lag2                |r| = 0.924
  TARGET_BPA_1.01_lag1                |r| = 0.924
  TARGET_BPA_1_lag1                   |r| = 0.911
  TARGET_BPP_2_lag1                   |r| = 0.911
  TARGET_DRE_3.11_lag1                |r|

2026-05-07 15:14:06 | INFO     | Prospectivo salvo: 76 obs | dataset_prospectivo.parquet


## Etapa 11 — Persistência dos artefatos para o Script 3

In [16]:
TARGETS_VALIDOS  = [t for t in TARGET_COLS if t in dataset.columns]
grupos_treino    = treino['CNPJ_CIA'].values

artefatos_df = {
    'dataset_preparado': dataset,
    'treino':            treino,
    'teste':             teste,
    'prospectivo':       prospectivo,   # Script 5: ITR Q1/2026 real + previsão Q2/Q3/DFP 2026
}
artefatos_meta = {
    'features'      : FEATURES_SELECIONADAS,
    'targets'            : TARGETS_VALIDOS,
    'targets_por_horizonte': {
        sufixo: [t for t in TARGETS_VALIDOS if t.endswith(sufixo)]
        for sufixo in ['_ITR_T1', '_ITR_T2', '_ITR_T3', '_DFP']
    },
    'target_cols_source' : TARGET_COLS_SOURCE,
    'kpis'          : kpis_presentes,
    'cols_setor'    : cols_setor,
    'cols_yoy'      : cols_yoy,
    'cols_lag':      cols_lag,
    'cols_razoes':    cols_razoes,       # V7: razões cruzadas
    'cols_interacao': cols_interacao,   # V7: interações setor×macro
    'cols_proporcao': cols_proporcao,   # V9: crescimento relativo
    'cols_pos_setor': cols_pos_setor,   # V9: posição no setor
    'cols_tri':       cols_tri,         # V9: sazonalidade trimestral
    'grupos_treino': grupos_treino,
    'params': {
        'versao'              : 'V9_FeatsRicas',
        'limiar_nulo'         : LIMIAR_NULO,
        'fator_winsor'        : FATOR_WINSOR,
        'max_cols_yoy'        : MAX_COLS_YOY,
        'clip_yoy'            : CLIP_YOY,
        'corr_min'            : CORR_MIN,
        'n_features_rfe'      : N_FEATURES_RFE,
        'frac_treino'         : FRAC_TREINO,
        'gap_yoy_min'         : GAP_YOY_MIN,
        'gap_yoy_max'         : GAP_YOY_MAX,
        'estrategia_target'   : 'proximo_dfp_estritamente_posterior',
        'pandas_version'      : pd.__version__,
        'macro_ano_inicio'    : ANO_INICIO_MACRO,
        'macro_ano_fim'       : ANO_FIM_MACRO,
    },
}

for nome, df_art in artefatos_df.items():
    cam = PASTA_SAIDA / f'{nome}.parquet'
    df_art.to_parquet(cam, index=False)
    logger.info("Salvo: %s | %d × %d | %.0f KB", cam.name, *df_art.shape, cam.stat().st_size/1024)

for nome, obj in artefatos_meta.items():
    cam = PASTA_SAIDA / f'{nome}.pkl'
    with open(cam,'wb') as f: pickle.dump(obj, f)
    logger.info("Salvo: %s", cam.name)

relatorio = {
    'versao'                   : 'V9_FeatsRicas',
    'estrategia_target'        : 'proximo_dfp_estritamente_posterior',
    'dataset_empresas'         : int(dataset['CNPJ_CIA'].nunique()),
    'dataset_anos'             : sorted(dataset['ANO'].dropna().astype(int).unique().tolist()),
    'n_obs_dfp'                : int((dataset['ORIGEM']=='DFP').sum()),
    'n_obs_itr'                : int((dataset['ORIGEM']=='ITR').sum()),
    'n_obs_total'              : int(len(dataset)),
    'n_obs_treino'             : int(len(treino)),
    'n_obs_treino_dfp'         : int((treino['ORIGEM']=='DFP').sum()),
    'n_obs_treino_itr'         : int((treino['ORIGEM']=='ITR').sum()),
    'n_obs_teste'              : int(len(teste)),
    'n_obs_prospectivo'        : int(len(prospectivo)),
    'anos_prospectivo'         : sorted(prospectivo['ANO'].dropna().astype(int).unique().tolist()) if not prospectivo.empty else [],
    'n_features_candidatas'    : int(len(FEATURES_CANDIDATAS)),
    'n_features_selecionadas'  : int(len(FEATURES_SELECIONADAS)),
    'targets'                  : TARGETS_VALIDOS,
    'features'                 : FEATURES_SELECIONADAS,
    'params'                   : artefatos_meta['params'],
    'nota_groupkfold'          : 'Usar GroupKFold(groups=grupos_treino) no Script 3.',
    'bugs_corrigidos'          : [
        'CAUSA RAIZ cobertura 0%: pd.DataFrame(series_bcb) misturava BCB mensal com EWZ '
            'diário — índice dominado por datas diárias deixava ipca/pib como NaN no '
            'merge_asof. Corrigido com merge_asof individual por série.',
        'BCB coleta ano a ano (loop 2015–2025): evita 406/413 por payload e delimita '
            'o intervalo ao período do TCC.',
        'EWZ tz_convert(None): índice UTC normalizado antes de entrar no dict.',
        'TZ_DATASET=None guard na Etapa 8: evita TypeError com dtype string inválido.',
        'aceleracao_ebitda duplicada: removido ternário explícito; dict.fromkeys() garante unicidade.',
        'Etapa 5: groupby.transform() em vez de apply() — preserva SETOR no pandas 3.x.',
        'Etapa 10: calcular_split retorna Series — preserva CNPJ_CIA no pandas 3.x.',
    ],
}
cam_rel = PASTA_SAIDA / 'logs' / 'auditoria_preparacao.json'
with open(cam_rel,'w',encoding='utf-8') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)

print("\n" + "═"*65)
print("  RESUMO FINAL — Script 2 V6 (DFP + ITR)")
print("═"*65)
print(f"  Total obs     : {len(dataset)} (DFP: {(dataset['ORIGEM']=='DFP').sum()} | ITR: {(dataset['ORIGEM']=='ITR').sum()})")
print(f"  Empresas      : {dataset['CNPJ_CIA'].nunique()} / 25")
print(f"  Treino (≤2022): {len(treino)} obs ({len(treino)/len(dataset):.0%})")
print(f"    ↳ DFP       : {(treino['ORIGEM']=='DFP').sum()}")
print(f"    ↳ ITR       : {(treino['ORIGEM']=='ITR').sum()}")
print(f"  Teste (23–24) : {len(teste)} obs ({len(teste)/len(dataset):.0%})")
print(f"  Prospectivo   : {len(prospectivo)} obs ({len(prospectivo)/len(dataset):.0%}) → Script 5")
print(f"  Features      : {len(FEATURES_SELECIONADAS)} selecionadas")
print(f"  Targets       : {TARGETS_VALIDOS}")
print(f"  Vs. V2(DFP)   : {len(dataset)/74:.1f}× mais observações")
print("═"*65)
print("  ⚠️  Script 3: usar GroupKFold(groups=grupos_treino) no CV")
print(f"  V7 adições        : YoY sem exclusões | lags macro | razões cruzadas | interações setor×macro | seleção multi-target")
print("  ✅  Pronto para o Script 3 (03_cvm_treino_v3.ipynb)")
print("═"*65)


2026-05-07 15:14:15 | INFO     | Salvo: dataset_preparado.parquet | 987 × 681 | 3229 KB
2026-05-07 15:14:16 | INFO     | Salvo: treino.parquet | 713 × 681 | 2435 KB
2026-05-07 15:14:16 | INFO     | Salvo: teste.parquet | 168 × 681 | 922 KB
2026-05-07 15:14:16 | INFO     | Salvo: prospectivo.parquet | 76 × 681 | 632 KB
2026-05-07 15:14:16 | INFO     | Salvo: features.pkl
2026-05-07 15:14:16 | INFO     | Salvo: targets.pkl
2026-05-07 15:14:16 | INFO     | Salvo: kpis.pkl
2026-05-07 15:14:16 | INFO     | Salvo: cols_setor.pkl
2026-05-07 15:14:16 | INFO     | Salvo: cols_yoy.pkl
2026-05-07 15:14:16 | INFO     | Salvo: cols_lag.pkl
2026-05-07 15:14:16 | INFO     | Salvo: grupos_treino.pkl
2026-05-07 15:14:16 | INFO     | Salvo: params.pkl



═════════════════════════════════════════════════════════════════
  RESUMO FINAL — Script 2 V6 (DFP + ITR)
═════════════════════════════════════════════════════════════════
  Total obs     : 987 (DFP: 230 | ITR: 757)
  Empresas      : 25 / 25
  Treino (≤2022): 713 obs (72%)
    ↳ DFP       : 181
    ↳ ITR       : 532
  Teste (23–24) : 168 obs (17%)
  Prospectivo   : 76 obs (8%) → Script 5
  Features      : 15 selecionadas
  Targets       : ['TARGET_DRE_3.01', 'TARGET_DRE_3.11', 'TARGET_EBITDA', 'TARGET_BPA_1', 'TARGET_BPA_1.01', 'TARGET_BPP_2.01', 'TARGET_BPP_2.03', 'TARGET_BPP_2', 'TARGET_DFC_MI_6.01']
  Vs. V2(DFP)   : 13.3× mais observações
═════════════════════════════════════════════════════════════════
  ⚠️  Script 3: usar GroupKFold(groups=grupos_treino) no CV
  ✅  Pronto para o Script 3 (03_cvm_modelagem.ipynb)
═════════════════════════════════════════════════════════════════
